<a href="https://colab.research.google.com/github/intisariapps-com/intiVoice_Studio/blob/main/intiVoice_Studio_WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎙️ intiVoice AI — Web Studio Visual Mandiri (Next.js & Cloudflare Edition V1.6.4)

> ⏰ **Terakhir Diperbarui:** 22 September 2026 | **Rilis:** V1.6.4 (Next.js SPA Embedded & Zero-Pip Protected)
> 🌟 **Antarmuka Produksi Penuh (Next.js 15):** Menghadirkan Web Studio resmi dengan 4 Tab lengkap (Studio Generasi, Produksi Naskah, Pustaka Suara, dan Pengaturan Mesin).
> 🌐 **Public HTTPS Cloudflare Tunnel:** Menghasilkan tautan publik aman (`https://xxxx.trycloudflare.com`) dengan proteksi SSL/TLS tanpa batas waktu.
> ⚡ **Zero-Configuration Connection (Same-Origin):** Frontend web otomatis mendeteksi server backend vokal GPU tanpa perlu salin-tempel URL secara manual!
> 🛡️ **Protected Zero-Pip Engine:** Memuat aset web dan biner terenkripsi PyArmor `intivoice_studio_engine.zip` (pemuatan < 0.5 detik).

---
### 🚀 Cara Menjalankan:
1. Pastikan runtime GPU aktif (**Runtime** → **Change runtime type** → **T4 GPU**).
2. (Opsional) Masukkan **Cloudflare Tunnel Token** jika memiliki domain tetap, atau biarkan kosong untuk Quick Tunnel gratis.
3. Klik tombol **Play (▶)** pada sel kode di bawah ini.
4. Tunggu ~40 detik hingga model siap. Klik tautan Cloudflare Tunnel yang tercetak di layar untuk membuka Web Studio di **Tab Browser Baru (Layar Penuh)**.


In [ ]:
"""
🎙️ INTIVOICE AI — WEB STUDIO VISUAL NEXT.JS (V1.6.4 PROTECTED ZERO-PIP)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
Menyajikan Antarmuka Produksi Penuh Next.js + Backend Vokal GPU 48kHz.
"""

# @title ⚙️ KONFIGURASI WEB STUDIO & CLOUDFLARE TUNNEL
# @markdown Masukkan token Cloudflare Named Tunnel atau Hugging Face (opsional):
CLOUDFLARE_TUNNEL_TOKEN = ""  # @param {type:"string"}
CLOUDFLARE_TUNNEL_DOMAIN = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### ⏱️ Pengaman Hemat GPU (Auto-Shutdown Idle Watchdog)
# @markdown Matikan runtime Colab otomatis jika tidak ada aktivitas sintesis baru (0 = Nonaktif).
AUTO_SHUTDOWN_MINUTES = 5  # @param [0, 1, 2, 3, 5, 10, 15, 30] {type:"raw"}

import os
import sys
import time
import subprocess
import zipfile
import sysconfig

print("=" * 80)
print("🎙️ MEMULAI INTIVOICE AI WEB STUDIO V1.6.4 (NEXT.JS & ZERO-PIP PROTECTED)")
print("=" * 80)

# 1. Download binary cloudflared jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ Mengunduh Cloudflare Tunnel client...")
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

# 2. Instal dependensi sistem dasar
print("📦 Memeriksa dependensi audio & server...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "fastapi", "uvicorn", "soundfile", "torchaudio", "pydantic", "requests"], check=False)

# 3. Unduh & pasang modul engine terenkripsi resmi (Zero-Pip & 100% Kebal Cache)
print("🔐 Mengunduh modul engine & Web Studio Next.js: intivoice_studio_engine.zip (V1.6.4)...")
pkg_url = f"https://raw.githubusercontent.com/intisariapps-com/intiVoice_Studio/main/intivoice_studio_engine.zip?t={int(time.time())}"
pkg_local = "/tmp/intivoice_studio_engine.zip"

subprocess.run(["wget", "-q", "--no-cache", "--no-cookies", "-O", pkg_local, pkg_url], check=True)
pkg_size_kb = round(os.path.getsize(pkg_local) / 1024, 1) if os.path.exists(pkg_local) else 0.0
print(f"📦 Paket Biner Terpasang: intivoice_studio_engine.zip (Versi V1.6.4 | {pkg_size_kb} KB)")

# Ekstrak paket ke folder site-packages Python (Zero-Pip Deployment)
site_pkg = sysconfig.get_paths()["purelib"]
with zipfile.ZipFile(pkg_local, "r") as zf:
    zf.extractall(site_pkg)

# Kebal Cache: Bersihkan registri modul Python agar selalu memuat modul baru dari disk
for mod in list(sys.modules.keys()):
    if "intivoice_studio_engine" in mod or "pyarmor" in mod:
        del sys.modules[mod]

import intivoice_studio_engine
from intivoice_studio_engine import run_studio_server
engine_ver = getattr(intivoice_studio_engine, "__version__", "1.6.4")
print(f"🚀 Modul Terverifikasi: intiVoice Studio Engine v{engine_ver} (Next.js SPA Embedded)")

# Jalankan Web Studio & Server GPU
run_studio_server(
    tunnel_token=CLOUDFLARE_TUNNEL_TOKEN,
    tunnel_domain=CLOUDFLARE_TUNNEL_DOMAIN,
    hf_token=HF_TOKEN,
    auto_shutdown_minutes=AUTO_SHUTDOWN_MINUTES,
    wait_forever=False
)
